# Current-API TIFF reconstruction smoke workflow

A hermetic, executable example of TIFF ingestion, phase preparation, reconstruction, and slice output.

In [1]:
from pathlib import Path
import shutil
import sys
import tempfile
from uuid import uuid4

repo_root = Path.cwd().resolve()
if not (repo_root / "lftomo").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import h5py
import numpy as np
import tifffile as tf
from skimage.transform import iradon

from lftomo.util.io import read_window, write_slice
from lftomo.util.log import log
from lftomo.util.mem import SharedNPArray
from lftomo.util.shapes import ProjOrder, ReconOrder, Y
from lftomo.util.window import CTFov, TomoDataSet


In [2]:
work_root = Path(tempfile.mkdtemp(prefix="lftomo-notebook-"))
log.set_logdir(work_root)
size = 16
n_projections = 12

y, x = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
phantom = (((x / 0.65) ** 2 + (y / 0.9) ** 2) <= 1).astype(np.float32)
projections = np.stack([
    0.2 + 0.8 * np.roll(phantom, shift=i % 4, axis=1)
    for i in range(n_projections)
]).astype(np.float32)
projections.shape


(12, 16, 16)

In [3]:
source = work_root / "projections"
source.mkdir()
source_files = []
for index, image in enumerate(projections):
    path = source / f"projection_{index:04d}.tif"
    tf.imwrite(path, image)
    source_files.append(path)

dataset = TomoDataSet.from_disk(
    source,
    CTFov(Theta=None, Degrees=180, Count=1),
    ProjOrder,
    Y(size),
    work_dtype=np.float32,
)


In [4]:
window = next(dataset.iter_with(substep=Y(1)))
projection_memory = SharedNPArray(
    f"notebook-projections-{uuid4().hex}",
    window.shape,
    dataset.dtype,
    allocate=True,
)
read_window(window, source_files, projection_memory, thread_max=2)

with projection_memory as loaded:
    np.testing.assert_allclose(loaded, projections)
window.shape


TIME  |RW START            |00:00:00.053155|0002.00GB|0058.00GB|""


TIME  |RW FILE INFO        |00:00:00.055694|0002.00GB|0058.00GB|""


TIME  |RW INDICIES         |00:00:00.061993|0002.00GB|0058.00GB|""


TIME  |RW ADDR             |00:00:00.064046|0002.00GB|0058.00GB|""


TIME  |RW ZIP              |00:00:00.066027|0002.00GB|0058.00GB|""


TIME  |RW READ             |00:00:00.090138|0002.00GB|0058.00GB|""


ProjOrder(Theta=12, Y=16, X=16)

In [5]:
with projection_memory as loaded:
    prepared = -np.log(np.clip(loaded.copy(), 1e-6, None))

theta_degrees = np.rad2deg(dataset.angles)
reconstruction = np.stack([
    iradon(prepared[:, row, :].T, theta=theta_degrees, circle=False, output_size=size)
    for row in range(size)
]).astype(np.float32)
reconstruction.shape


(16, 16, 16)

In [6]:
recon_memory = SharedNPArray(
    f"notebook-recon-{uuid4().hex}",
    ReconOrder(Y=size, Z=size, X=size),
    np.float32,
    allocate=True,
)
with recon_memory as target:
    target[:] = reconstruction

output_path = work_root / "reconstruction_slice.tif"
write_slice(recon_memory, output_path, i=size // 2)
assert output_path.is_file()
tf.imread(output_path).shape


(16, 16)

In [7]:
projection_memory.shutdown()
recon_memory.shutdown()
shutil.rmtree(work_root)
"smoke workflow complete"


'smoke workflow complete'